# Notebook 56 — Latent Coordinate Transformation and Global Law

Notebook 55 suggested that residual-field corrections are not the right next layer.
The stronger hypothesis is that structure may live in the **coordinate system** rather than
in a large residual correction.

We test global laws on transformed coordinates:

\[
(r,c) \mapsto \phi(r,c)
\]

followed by a shared parametric law

\[
g(r,c) \approx h(\phi(r,c))
\]

## Main goals

1. Compare the baseline global polynomial law on `(r,c)` against richer coordinate transforms.
2. Test hand-crafted coordinate augmentations.
3. Test low-dimensional latent coordinate transforms.
4. Evaluate harder blocks using transformed global laws.
5. See whether representation change improves where residual corrections did not.

In [ ]:
# Core imports

import warnings
warnings.filterwarnings("ignore")

import os
import glob
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression, Ridge

try:
    from IPython.display import display
except Exception:
    display = print

np.random.seed(42)

In [ ]:
# Data discovery and synthetic fallback

DATA_PATH = None

def autodetect_data_path():
    candidates = []
    patterns = [
        "*residual*flow*.parquet",
        "*residual*flow*.csv",
        "*governing*flow*.parquet",
        "*governing*flow*.csv",
        "*.parquet",
        "*.csv",
        "*.pkl",
        "*.pickle",
    ]
    for pat in patterns:
        candidates.extend(glob.glob(pat))
        candidates.extend(glob.glob(os.path.join("data", pat)))
        candidates.extend(glob.glob(os.path.join("outputs", pat)))
    candidates = [c for c in candidates if os.path.isfile(c)]
    return candidates[0] if candidates else None

def synthetic_dataset():
    systems = ["entropy", "unevenness"]
    tasks = ["zeta_vs_gue", "zeta_vs_poisson"]
    forcing_modes = ["capacity_gap", "feature_gap", "condition_gap"]
    ks = [3, 5, 7]
    flow_modes = ["linear", "nonlinear"]

    rows = []
    sample_id = 0
    for system in systems:
        for task in tasks:
            for forcing_mode in forcing_modes:
                for k in ks:
                    for flow_mode in flow_modes:
                        n_paths = 14
                        n_steps = 42
                        c_grid = np.linspace(-1.25, 1.05, n_steps)

                        sys_shift = 0.06 if system == "entropy" else -0.04
                        task_shift = 0.05 if task == "zeta_vs_gue" else -0.03
                        force_shift = {"capacity_gap": 0.00, "feature_gap": 0.03, "condition_gap": 0.08}[forcing_mode]
                        k_shift = {3: -0.05, 5: 0.02, 7: 0.06}[k]
                        flow_shift = 0.05 if flow_mode == "nonlinear" else -0.02
                        nl_gain = 1.0 if flow_mode == "nonlinear" else 0.55

                        for path_id in range(n_paths):
                            r = -0.75 + 0.10 * path_id + 0.05 * np.sin(0.7 * path_id)
                            for window_id, c in enumerate(c_grid):
                                g = (
                                    0.58 * np.tanh(1.35 * c)
                                    + 0.42 * c
                                    - 0.78 * r
                                    + 0.20 * r**2
                                    + nl_gain * 0.07 * c**2
                                    + nl_gain * 0.10 * r * c
                                    - nl_gain * 0.025 * r**3
                                    + sys_shift + task_shift + force_shift + k_shift + flow_shift
                                )
                                if forcing_mode == "condition_gap":
                                    g += 0.06 * np.sin(2.3 * c)
                                if system == "entropy":
                                    g += 0.03 * np.cos(1.2 * c)
                                if task == "zeta_vs_poisson":
                                    g -= 0.015 * c
                                if flow_mode == "linear":
                                    g -= 0.09 * r**2
                                    g += 0.015 * c * r

                                delta_c = c_grid[min(window_id + 1, n_steps - 1)] - c if window_id < n_steps - 1 else c - c_grid[max(window_id - 1, 0)]
                                noise = 0.012 * np.random.randn()
                                next_r = r + (g + noise) * delta_c

                                rows.append(
                                    {
                                        "system": system,
                                        "task": task,
                                        "forcing_mode": forcing_mode,
                                        "k": k,
                                        "flow_mode": flow_mode,
                                        "condition_coord": c,
                                        "residual": r,
                                        "predicted_flow": g + noise,
                                        "next_residual": next_r,
                                        "delta_condition": delta_c,
                                        "sample_id": sample_id,
                                        "path_id": path_id,
                                        "window_id": window_id,
                                    }
                                )
                                r = next_r
                                sample_id += 1
    return pd.DataFrame(rows)

if DATA_PATH is None:
    DATA_PATH = autodetect_data_path()

USE_SYNTHETIC_FALLBACK = DATA_PATH is None
print("Selected DATA_PATH:", DATA_PATH)
print("USE_SYNTHETIC_FALLBACK:", USE_SYNTHETIC_FALLBACK)

In [ ]:
# Loading + schema alignment

def load_dataframe(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".parquet":
        return pd.read_parquet(path)
    if ext in [".pkl", ".pickle"]:
        return pd.read_pickle(path)
    return pd.read_csv(path)

alias_groups = {
    "condition_coord": ["condition_coord", "c", "condition", "cond", "condition_coordinate"],
    "residual": ["residual", "r", "resid"],
    "predicted_flow": ["predicted_flow", "flow", "g", "drdc", "delta_residual_per_condition"],
    "next_residual": ["next_residual", "r_next", "next_r"],
    "delta_condition": ["delta_condition", "dc", "delta_c"],
    "forcing_mode": ["forcing_mode", "forcing", "forcing_gap", "mode"],
}

def align_schema(df):
    cols = list(df.columns)
    rename_map = {}
    for canonical, aliases in alias_groups.items():
        for a in aliases:
            if a in cols and a != canonical:
                rename_map[a] = canonical
                break
    df = df.rename(columns=rename_map)

    if "next_residual" not in df.columns and {"residual", "predicted_flow", "delta_condition"}.issubset(df.columns):
        df["next_residual"] = df["residual"] + df["predicted_flow"] * df["delta_condition"]

    if "delta_condition" not in df.columns and "condition_coord" in df.columns:
        df = df.sort_values(["condition_coord"]).copy()
        dc = np.gradient(df["condition_coord"].to_numpy())
        df["delta_condition"] = dc

    defaults = {
        "system": "unknown_system",
        "task": "unknown_task",
        "forcing_mode": "unknown_forcing",
        "k": 5,
        "flow_mode": "unknown_flow",
        "sample_id": np.arange(len(df)),
        "path_id": 0,
        "window_id": np.arange(len(df)),
    }
    for k, v in defaults.items():
        if k not in df.columns:
            df[k] = v

    required = ["condition_coord", "residual", "predicted_flow"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns after alignment: {missing}")
    return df

if USE_SYNTHETIC_FALLBACK:
    df = synthetic_dataset()
    data_source = "synthetic_fallback"
else:
    df = align_schema(load_dataframe(DATA_PATH))
    data_source = DATA_PATH

df = align_schema(df)
print("Data source:", data_source)
print("Shape:", df.shape)
display(df.head())

In [ ]:
# Regime dataset and metadata

group_cols = ["system", "task", "forcing_mode", "k", "flow_mode"]
regime_rows = []
regime_subsets = {}

for vals, sub in df.groupby(group_cols):
    if len(sub) < 30:
        continue
    kval = int(vals[3]) if float(vals[3]).is_integer() else vals[3]
    regime_id = f"{vals[0]}|{vals[1]}|{vals[2]}|k={kval}|{vals[4]}"
    regime_rows.append({
        "regime_id": regime_id,
        "system": vals[0],
        "task": vals[1],
        "forcing_mode": vals[2],
        "k": float(vals[3]),
        "flow_mode": vals[4],
    })
    regime_subsets[regime_id] = sub.copy()

coef_df = pd.DataFrame(regime_rows).reset_index(drop=True)

meta_df = coef_df[["regime_id", "system", "task", "forcing_mode", "flow_mode", "k"]].copy()
X_meta = pd.get_dummies(meta_df[["system", "task", "forcing_mode", "flow_mode"]], drop_first=False)
X_meta["k"] = coef_df["k"].astype(float).values
X_meta["k2"] = coef_df["k"].astype(float).values**2

display(coef_df.head())
display(X_meta.head())

## Coordinate transforms

In [ ]:
# Coordinate transforms and global field models

poly3 = PolynomialFeatures(degree=3, include_bias=True)

def make_coords(sub, mode="baseline"):
    r = sub["residual"].to_numpy(dtype=float)
    c = sub["condition_coord"].to_numpy(dtype=float)

    if mode == "baseline":
        return np.column_stack([r, c])

    if mode == "handcrafted":
        return np.column_stack([
            r,
            c,
            np.tanh(c),
            r * c,
            np.log1p(r**2),
            c**2,
            np.sin(c),
        ])

    if mode == "scaled":
        return np.column_stack([
            r,
            c,
            r / (1.0 + np.abs(c)),
            c / (1.0 + np.abs(r)),
            np.tanh(r),
            np.tanh(c),
        ])

    raise ValueError(f"Unknown mode: {mode}")

def fit_global_field_model(train_df, coord_mode="baseline", use_ridge=True):
    X_list = []
    y_list = []

    for rid in train_df["regime_id"].tolist():
        sub = regime_subsets[rid]
        coords = make_coords(sub, mode=coord_mode)
        Phi = poly3.fit_transform(coords)
        X_list.append(Phi)
        y_list.append(sub["predicted_flow"].to_numpy(dtype=float))

    X = np.vstack(X_list)
    y = np.concatenate(y_list)

    model = Ridge(alpha=1.0) if use_ridge else LinearRegression()
    model.fit(X, y)
    return model

def predict_field_model(model, sub, coord_mode="baseline"):
    coords = make_coords(sub, mode=coord_mode)
    Phi = poly3.fit_transform(coords)
    return model.predict(Phi)

def trajectory_gap_from_pred(sub, pred_ref, pred_cmp, n_r0=15):
    cmin, cmax = sub["condition_coord"].min(), sub["condition_coord"].max()
    rmin, rmax = sub["residual"].min(), sub["residual"].max()
    cgrid = np.linspace(cmin, cmax, 40)
    r0s = np.linspace(np.quantile(sub["residual"], 0.05), np.quantile(sub["residual"], 0.95), n_r0)
    flow_cap = max(1.0, 2.5 * np.quantile(np.abs(sub["predicted_flow"]), 0.995))

    # simple surrogate: train local interpolants from sampled (r,c)->g values on this regime subset
    pts = sub[["residual", "condition_coord"]].to_numpy(dtype=float)

    def nearest_field(pred_values):
        pred_values = np.asarray(pred_values, dtype=float)
        def f(r, c):
            q = np.array([r, c], dtype=float)
            j = np.argmin(np.sum((pts - q[None, :])**2, axis=1))
            return float(np.clip(pred_values[j], -flow_cap, flow_cap))
        return f

    f_ref = nearest_field(pred_ref)
    f_cmp = nearest_field(pred_cmp)

    def integrate(field_fn, r0):
        r = float(np.clip(r0, rmin, rmax))
        traj = [r]
        for i in range(len(cgrid) - 1):
            c = cgrid[i]
            dc = float(cgrid[i+1] - cgrid[i])
            g = float(np.clip(field_fn(r, c), -flow_cap, flow_cap))
            r = float(np.clip(r + g * dc, rmin - 0.5, rmax + 0.5))
            traj.append(r)
        return np.array(traj)

    errs = []
    for r0 in r0s:
        t_ref = integrate(f_ref, r0)
        t_cmp = integrate(f_cmp, r0)
        errs.append(math.sqrt(mean_squared_error(t_ref, t_cmp)))
    return float(np.mean(errs))

## Latent coordinate transform

Use a global PCA-based transform of a richer handcrafted coordinate bank.

In [ ]:
# Learn a latent 2D coordinate transform from richer handcrafted coordinates

coord_bank = []
for rid in coef_df["regime_id"].tolist():
    sub = regime_subsets[rid]
    coord_bank.append(make_coords(sub, mode="handcrafted"))
coord_bank = np.vstack(coord_bank)

coord_scaler = StandardScaler()
coord_bank_std = coord_scaler.fit_transform(coord_bank)

coord_pca = PCA(n_components=2, random_state=42)
coord_pca.fit(coord_bank_std)

def make_latent_coords(sub):
    coords = make_coords(sub, mode="handcrafted")
    coords_std = coord_scaler.transform(coords)
    return coord_pca.transform(coords_std)

In [ ]:
def fit_latent_field_model(train_df, use_ridge=True):
    X_list = []
    y_list = []

    for rid in train_df["regime_id"].tolist():
        sub = regime_subsets[rid]
        coords = make_latent_coords(sub)
        Phi = poly3.fit_transform(coords)
        X_list.append(Phi)
        y_list.append(sub["predicted_flow"].to_numpy(dtype=float))

    X = np.vstack(X_list)
    y = np.concatenate(y_list)

    model = Ridge(alpha=1.0) if use_ridge else LinearRegression()
    model.fit(X, y)
    return model

def predict_latent_field_model(model, sub):
    coords = make_latent_coords(sub)
    Phi = poly3.fit_transform(coords)
    return model.predict(Phi)

## Compare transformed global laws on harder blocks

In [ ]:
hard_block_masks = {
    "k_extrapolate_high": coef_df["k"].astype(float) == 7.0,
    "k_extrapolate_low": coef_df["k"].astype(float) == 3.0,
    "system_holdout::entropy": coef_df["system"].astype(str) == "entropy",
    "system_holdout::unevenness": coef_df["system"].astype(str) == "unevenness",
}

rows = []

for block_name, test_mask in hard_block_masks.items():
    train_mask = ~test_mask
    train_df = coef_df.loc[train_mask].reset_index(drop=True)
    test_df = coef_df.loc[test_mask].reset_index(drop=True)

    model_baseline = fit_global_field_model(train_df, coord_mode="baseline", use_ridge=True)
    model_hand = fit_global_field_model(train_df, coord_mode="handcrafted", use_ridge=True)
    model_scaled = fit_global_field_model(train_df, coord_mode="scaled", use_ridge=True)
    model_latent = fit_latent_field_model(train_df, use_ridge=True)

    for rid in test_df["regime_id"].tolist():
        sub = regime_subsets[rid]
        y_true = sub["predicted_flow"].to_numpy(dtype=float)

        pred_baseline = predict_field_model(model_baseline, sub, coord_mode="baseline")
        pred_hand = predict_field_model(model_hand, sub, coord_mode="handcrafted")
        pred_scaled = predict_field_model(model_scaled, sub, coord_mode="scaled")
        pred_latent = predict_latent_field_model(model_latent, sub)

        rows.append({
            "block": block_name,
            "regime_id": rid,
            "field_rmse_baseline": math.sqrt(mean_squared_error(y_true, pred_baseline)),
            "field_rmse_handcrafted": math.sqrt(mean_squared_error(y_true, pred_hand)),
            "field_rmse_scaled": math.sqrt(mean_squared_error(y_true, pred_scaled)),
            "field_rmse_latent": math.sqrt(mean_squared_error(y_true, pred_latent)),

            "traj_rmse_handcrafted_vs_baseline": trajectory_gap_from_pred(sub, pred_baseline, pred_hand),
            "traj_rmse_scaled_vs_baseline": trajectory_gap_from_pred(sub, pred_baseline, pred_scaled),
            "traj_rmse_latent_vs_baseline": trajectory_gap_from_pred(sub, pred_baseline, pred_latent),
        })

compare_df = pd.DataFrame(rows)
display(compare_df.head())

In [ ]:
summary_df = compare_df.groupby("block")[[
    "field_rmse_baseline",
    "field_rmse_handcrafted",
    "field_rmse_scaled",
    "field_rmse_latent",
    "traj_rmse_handcrafted_vs_baseline",
    "traj_rmse_scaled_vs_baseline",
    "traj_rmse_latent_vs_baseline",
]].mean().reset_index()

display(summary_df)

In [ ]:
# Field RMSE comparison

fig, axes = plt.subplots(1, 2, figsize=(16, 4))
x = np.arange(len(summary_df))
w = 0.20

axes[0].bar(x - 1.5*w, summary_df["field_rmse_baseline"], width=w, label="baseline")
axes[0].bar(x - 0.5*w, summary_df["field_rmse_handcrafted"], width=w, label="handcrafted")
axes[0].bar(x + 0.5*w, summary_df["field_rmse_scaled"], width=w, label="scaled")
axes[0].bar(x + 1.5*w, summary_df["field_rmse_latent"], width=w, label="latent")
axes[0].set_xticks(x)
axes[0].set_xticklabels(summary_df["block"], rotation=30, ha="right")
axes[0].set_title("Field RMSE by harder block")
axes[0].legend()

axes[1].bar(x - w, summary_df["traj_rmse_handcrafted_vs_baseline"], width=w, label="handcrafted vs baseline")
axes[1].bar(x, summary_df["traj_rmse_scaled_vs_baseline"], width=w, label="scaled vs baseline")
axes[1].bar(x + w, summary_df["traj_rmse_latent_vs_baseline"], width=w, label="latent vs baseline")
axes[1].set_xticks(x)
axes[1].set_xticklabels(summary_df["block"], rotation=30, ha="right")
axes[1].set_title("Trajectory gap relative to baseline")
axes[1].legend()

plt.tight_layout()
plt.show()

## Representative regime comparison

In [ ]:
# pick the regime where latent has largest field RMSE gain over baseline

rep_idx = int(np.argmax(compare_df["field_rmse_baseline"] - compare_df["field_rmse_latent"]))
rep = compare_df.iloc[rep_idx]
regime_id = rep["regime_id"]
sub = regime_subsets[regime_id]
block_name = rep["block"]

train_mask = ~hard_block_masks[block_name]
train_df = coef_df.loc[train_mask].reset_index(drop=True)

model_baseline = fit_global_field_model(train_df, coord_mode="baseline", use_ridge=True)
model_hand = fit_global_field_model(train_df, coord_mode="handcrafted", use_ridge=True)
model_scaled = fit_global_field_model(train_df, coord_mode="scaled", use_ridge=True)
model_latent = fit_latent_field_model(train_df, use_ridge=True)

pred_baseline = predict_field_model(model_baseline, sub, coord_mode="baseline")
pred_hand = predict_field_model(model_hand, sub, coord_mode="handcrafted")
pred_scaled = predict_field_model(model_scaled, sub, coord_mode="scaled")
pred_latent = predict_latent_field_model(model_latent, sub)
y_true = sub["predicted_flow"].to_numpy(dtype=float)

fig, axes = plt.subplots(1, 5, figsize=(20, 4), sharey=True)

panels = [
    ("Empirical", y_true),
    ("Baseline", pred_baseline),
    ("Handcrafted", pred_hand),
    ("Scaled", pred_scaled),
    ("Latent", pred_latent),
]

for ax, (title, vals) in zip(axes, panels):
    sc = ax.scatter(sub["condition_coord"], sub["residual"], c=vals, s=18)
    ax.set_title(title)
    ax.set_xlabel("condition coordinate c")
    plt.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
axes[0].set_ylabel("residual")

fig.suptitle(f"Representative regime: {regime_id}", y=1.03)
plt.tight_layout()
plt.show()

## Final verdicts

In [ ]:
def verdict_row(row):
    best_alt = min(row["field_rmse_handcrafted"], row["field_rmse_scaled"], row["field_rmse_latent"])
    if best_alt < row["field_rmse_baseline"]:
        if row["field_rmse_latent"] <= row["field_rmse_handcrafted"] and row["field_rmse_latent"] <= row["field_rmse_scaled"]:
            return "latent transform best"
        if row["field_rmse_handcrafted"] <= row["field_rmse_scaled"]:
            return "handcrafted transform best"
        return "scaled transform best"
    return "baseline remains best"

summary_df["verdict"] = summary_df.apply(verdict_row, axis=1)
display(summary_df)

## Working conclusion

Notebook 56 tests whether regime structure is better handled by changing coordinates rather than by adding residual corrections.

A strong result is:
- transformed coordinates reduce field RMSE on harder blocks
- latent coordinate transforms outperform naive residual corrections
- representation change helps where coefficient-only or residual-only corrections plateau

If that pattern holds on your real exports, the next notebook should be:

**Notebook 57 — jointly learned regime-aware coordinate transforms**